# Colecting and combing Arome and Metar Dataset
## Objective
The objective of this notebook is to prepare a combined dataset containing meteorological variables from the AROME model and wind observations extracted from METAR messages. The final dataset may be used for exploratory analysis and for building machine-learning models to predict wind gusts .

In [9]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path
import polars as pl 
import matplotlib.pyplot as plt
from IPython.display import display


project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [10]:
Arome = pl.read_csv("../src/Preparing/Arome_clean_final.csv")
metar = pl.read_csv("../src/Preparing/Rafale_METAR.csv")


display(Arome.columns)
display(metar.columns)

['station',
 'datetime',
 'longitude',
 'latitude',
 'u10',
 'v10',
 't2m',
 'rh2m',
 'u850',
 'v850',
 'u950',
 'v950',
 'psurf',
 'u_gust60',
 'v_gust60',
 'tke20m',
 'edr20m',
 'pblh']

['time', 'indicatif', 'wind_mean', 'wind_final', 'has_gust']

### The AROME dataset contains the following meteorological variables: 

- station 
- datetime 
- longitude 
- latitude 
- u10 and v10 
- t2m - rh2m 
- u850 and v850 
- u950 and v950 
- psurf 
- u_gust60 and v_gust60 
- tke20m 
- edr20m 
- pblh 

### METAR Dataset The METAR dataset contains wind observations: 

- time 
- indicatif 
- wind_mean 
- wind_final 
- has_gust 

### Convert Numeric Station Codes to ICAO Codes

The AROME dataset uses numeric station identifiers, while the METAR dataset uses ICAO station codes.

In [ ]:

code_OACI = {
    60033: "GMML",
    60060: "GMMF",
    60096: "GMMH",
    60101: "GMTT",
    60107: "GMTA",
    60115: "GMFO",
    60120: "GMMP",
    60135: "GMME",
    60136: "GMSL",
    60141: "GMFF",
    60150: "GMFM",
    60155: "GMMC",
    60156: "GMMN",
    60160: "GMFI",
    60191: "GMMD",
    60200: "GMFB",
    60210: "GMFK",
    60220: "GMMI",
    60230: "GMMX",
    60250: "GMAA",
    60252: "GMAD",
    60265: "GMMZ",
    60280: "GMAG",
    60285: "GMAT",
    60340: "GMMW"
}

Arome = Arome.with_columns(
    pl.col("station")
    .cast(pl.Int64, strict=False)
    .replace_strict(
        code_OACI,
        default=None,
        return_dtype=pl.Utf8
    )
    .alias("indicatif")
)


## Station Identifier Harmonization 
AROME stations were represented by numeric codes, while METAR observations used ICAO identifiers. A mapping dictionary was used to create an `indicatif` column in the AROME dataset. This common identifier is required to merge both datasets.

In [16]:
print( Arome.select( pl.col("datetime").min().alias("start"), pl.col("datetime").max().alias("end") ) ) 
print("METAR period:") 
print( metar.select( pl.col("time").min().alias("start"), pl.col("time").max().alias("end") ) )

shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ start               ┆ end                 │
│ ---                 ┆ ---                 │
│ str                 ┆ str                 │
╞═════════════════════╪═════════════════════╡
│ 2019-01-01 21:00:00 ┆ 2025-12-31 23:00:00 │
└─────────────────────┴─────────────────────┘
METAR period:
shape: (1, 2)
┌────────────────────────────┬────────────────────────────┐
│ start                      ┆ end                        │
│ ---                        ┆ ---                        │
│ str                        ┆ str                        │
╞════════════════════════════╪════════════════════════════╡
│ 2021-10-01T00:00:00.000000 ┆ 2025-07-07T11:00:00.000000 │
└────────────────────────────┴────────────────────────────┘


## Compare Available Stations

In [17]:
arome_stations = set( Arome["indicatif"] .drop_nulls() .unique() .to_list() ) 
metar_stations = set( metar["indicatif"] .drop_nulls() .unique() .to_list() ) 
common_stations = arome_stations & metar_stations 
print("Number of AROME stations:", len(arome_stations)) 
print("Number of METAR stations:", len(metar_stations)) 
print("Number of common stations:", len(common_stations)) 
print("Common stations:", sorted(common_stations)) 
print("Only in AROME:", sorted(arome_stations - metar_stations)) 
print("Only in METAR:", sorted(metar_stations - arome_stations))

Number of AROME stations: 25
Number of METAR stations: 33
Number of common stations: 24
Common stations: ['GMAA', 'GMAD', 'GMAG', 'GMAT', 'GMFB', 'GMFF', 'GMFI', 'GMFK', 'GMFM', 'GMFO', 'GMMC', 'GMMD', 'GMME', 'GMMH', 'GMMI', 'GMML', 'GMMN', 'GMMP', 'GMMW', 'GMMX', 'GMMZ', 'GMSL', 'GMTA', 'GMTT']
Only in AROME: ['GMMF']
Only in METAR: ['GMAR', 'GMAZ', 'GMMA', 'GMMB', 'GMMG', 'GMMK', 'GMMT', 'GMTN', 'GMZB']


## Merge Conclusion

The AROME–METAR merging workflow was implemented in the script:

```text
src/preparing/join.py
```

The script performs the complete preparation process, including column correction, station-code conversion, datetime standardization, missing-value removal, duplicate handling, and wind-variable preparation.

AROME numeric station codes are converted into ICAO codes to match the METAR `indicatif` field. The two datasets are then merged using an inner join based on the station ICAO code and observation datetime.

The final dataset contains approximately 527,000 matched observations.

The final merged dataset is exported as:

```text
Dataset_AROME_METAR.csv
Dataset_AROME_METAR.parquet
```

Overall, `src/preparing/join.py` provides a reproducible pipeline for combining AROME predictors with METAR wind and gust observations.
